In [3]:
import faker
import os
import time

from datetime import datetime, timezone

import pandas as pd

import duckdb

from dotenv import load_dotenv

In [4]:
load_dotenv()

# Configuration
HOST = os.getenv("MQTT_HOST", "test.mosquitto.org")
PORT = int(os.getenv("MQTT_PORT", "1883"))
NAMESPACE = os.environ["MQTT_NAMESPACE"]

NUMBER_OF_SENSORS = int(os.environ["NUMBER_OF_SENSORS"])

In [5]:
fake = faker.Faker()

customers = []
# Generate data for each customer
for idx in range(NUMBER_OF_SENSORS):

    # Unique meter ID
    meter_serial = fake.bothify(text="MTR-####-????").upper()



    # Create a cust
    customers.append(
        {
            # Business Key
            "customer_id": f"CUST-{idx:05d}",
            # Customer attributes
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "address": fake.address().replace("\n", ", "),
            "email": fake.email(),
            # Meter assignment
            "meter_id": idx,
            # SCD2
            "valid_from": datetime.now(timezone.utc).isoformat(),
            "valid_to": None,
            "is_current": True,
        }
    )

customers_df = pd.DataFrame.from_records(customers)

display(customers_df)


,customer_id,first_name,last_name,address,email,meter_id,valid_from,valid_to,is_current
0,CUST-00000,Peter,Woods,"84919 Sara Views, Port Davidshire, ND 94112",coxcynthia@example.com,0,2026-11-24T10:21:27.982295+00:00,None,True
1,CUST-00001,Samuel,Smith,"350 Perez Brooks Suite 830, Salasberg, CA 99620",elizabeth77@example.org,1,2026-11-24T10:21:27.982827+00:00,None,True
2,CUST-00002,Ricardo,Lopez,"8355 Steven Trafficway, Perryfort, CT 87174",fboyle@example.com,2,2026-11-24T10:21:27.983315+00:00,None,True
3,CUST-00003,Kelly,Garrett,"187 Greer Light Suite 014, New Kathleen, WI 23495",pagejoshua@example.net,3,2026-11-24T10:21:27.984025+00:00,None,True
4,CUST-00004,Bradley,Ramirez,"2429 Sierra Land Suite 738, Bowmanstad, NJ 54925",leeangela@example.com,4,2026-11-24T10:21:27.984560+00:00,None,True


In [7]:

con = duckdb.connect()

meter_df = con.execute("""
    SELECT COUNT(*)
    FROM read_json_auto(
        '../data/*.jsonl',
        records=true
    )
""").df()

display(df)

,sensor_id,sequence_no,timestamp,cumulative_value,_topic,_received_at
0,0,7,2026-11-24 10:19:30.581252+00:00,2.1664,82734c1a3bc7/sensor/sensor_0,2026-11-24 10:19:30.596723+00:00
1,1,7,2026-11-24 10:19:30.582951+00:00,2.5105,82734c1a3bc7/sensor/sensor_1,2026-11-24 10:19:30.611401+00:00
2,2,7,2026-11-24 10:19:30.583515+00:00,3.1391,82734c1a3bc7/sensor/sensor_2,2026-11-24 10:19:30.611476+00:00
3,3,7,2026-11-24 10:19:30.584208+00:00,2.2393,82734c1a3bc7/sensor/sensor_3,2026-11-24 10:19:30.611514+00:00
4,4,7,2026-11-24 10:19:30.584868+00:00,4.6620,82734c1a3bc7/sensor/sensor_4,2026-11-24 10:19:30.611554+00:00
...,...,...,...,...,...,...
165,0,40,2026-11-24 10:20:03.671320+00:00,16.5749,82734c1a3bc7/sensor/sensor_0,2026-11-24 10:20:03.685625+00:00
166,1,40,2026-11-24 10:20:03.671757+00:00,19.1339,82734c1a3bc7/sensor/sensor_1,2026-11-24 10:20:03.715010+00:00
167,2,40,2026-11-24 10:20:03.672002+00:00,19.7167,82734c1a3bc7/sensor/sensor_2,2026-11-24 10:20:03.715097+00:00
168,3,40,2026-11-24 10:20:03.672310+00:00,17.7490,82734c1a3bc7/sensor/sensor_3,2026-11-24 10:20:03.715136+00:00


In [20]:
columns = ",".join(["first_name", "last_name", "meter_id"])

con.execute(f"""
    SELECT 
        {columns}
    FROM
        customers_df
    ORDER BY
        last_name DESC
""").df()

,first_name,last_name,meter_id
0,Peter,Woods,0
1,Samuel,Smith,1
2,Bradley,Ramirez,4
3,Ricardo,Lopez,2
4,Kelly,Garrett,3


In [14]:
customers_df.sort_values(by=['last_name'], ascending=False)

,customer_id,first_name,last_name,address,email,meter_id,valid_from,valid_to,is_current
0,CUST-00000,Peter,Woods,"84919 Sara Views, Port Davidshire, ND 94112",coxcynthia@example.com,0,2026-11-24T10:21:27.982295+00:00,None,True
1,CUST-00001,Samuel,Smith,"350 Perez Brooks Suite 830, Salasberg, CA 99620",elizabeth77@example.org,1,2026-11-24T10:21:27.982827+00:00,None,True
4,CUST-00004,Bradley,Ramirez,"2429 Sierra Land Suite 738, Bowmanstad, NJ 54925",leeangela@example.com,4,2026-11-24T10:21:27.984560+00:00,None,True
2,CUST-00002,Ricardo,Lopez,"8355 Steven Trafficway, Perryfort, CT 87174",fboyle@example.com,2,2026-11-24T10:21:27.983315+00:00,None,True
3,CUST-00003,Kelly,Garrett,"187 Greer Light Suite 014, New Kathleen, WI 23495",pagejoshua@example.net,3,2026-11-24T10:21:27.984025+00:00,None,True
